In [23]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [24]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [25]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [11]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [10]:
len(documents)

72

In [26]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [27]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [47]:
import importlib
import rag_helper 
import ingest

importlib.reload(rag_helper)


documents = ingest.load_github_data()
index = ingest.build_index(documents)

assistant = rag_helper.RAGBase(
    index=index,
    llm_client=openai_client
)

answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print("Total tokens used: ", usage.total_tokens)

The loop keeps calling the model with a `while True` loop. After each response, it checks whether the model returned any `function_call` items:

- If there are function calls, the code runs the tools, appends the tool outputs to `messages`, and calls the model again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is: **no function calls in the latest response**.
Total tokens used:  7221


In [49]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [51]:
index = ingest.build_index(chunks)

assistant = rag_helper.RAGBase(
    index=index,
    llm_client=openai_client
)

answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print("Total tokens used: ", usage.total_tokens)

Total tokens used:  2404
